# M1P1 Creating and Managing Environments

In this module, environment management within Jupyter and within the HPC will be covered in a series of lectures. Follow along to create a conda environment that will be used for an R kernel within Jupyter for the rest of this week's Module. 

# In this practice
* [Environments](#env)
* [Conda / miniconda3](#cu)
* [Creation of environment from Module_1.yml file](#yml)
* [Creating an environment to fit your needs and global environments within the course](#make)

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

<a id='env'></a>
# Environments
An environment is a self-contained workspace within a computer, or in our case, the Hellbender HPC cluster, that includes specific software, libraries, and dependencies necessary for a particular task. In this course, we will utilize environments to run bioinformatics tools in a pre-packaged setup, making it simple and easy to interface with Jupyter Notebook and replicate across systems. These environments enable us to isolate programs and tools, preventing them from interfering with each other in cases where they require specific versions or dependencies. 

### Packages
Packages are collections of software, tools, libraries, and dependencies bundled together to provide specific functionality. For example, if we were to create an environment for processing RNA-Seq data, we might include tools such as `fastqc`, `samtools`, and `salmon`, among others. To create an environment for this module to generate a heatmap in R, we would include tools such as `r-base`, `r-tidyverse`, `r-irkernel`, `r-pheatmap`, and `bioconductor-deseq2`. 

### Channels
Channels are sources or repositories from which packages are stored or distributed. When you install a package, Conda, the environment manager we will use today, looks for it in these channels. In this course, most of the packages we use will be found in the `bioconda` channel, which hosts a wide range of bioinformatics tools. 

### Dependencies
When using an environment manager, many of the packages you install require other packages to function correctly. The packages that it depends upon are called dependencies. For example, a bioinformatics tool might require specific versions of Python libraries or system utilities. The environment manager will automatically identify and install all necessary dependencies from a channel into your environment, ensuring that the environment is functional and can be reproduced. 

This is essential for use in pipelines or techniques that are shared across labs, as well as in this class. The conda environments we will create and install will ensure that we have consistent working versions of software and tools, making analyses reproducible and trustworthy. 


<a id='cu'></a>
# Conda / Miniconda3 / Mamba
[Conda](https://docs.conda.io/en/latest) is an environment manager that allows for package and dependency management. Conda itself has many pre-installed packages and can occupy a significant amount of space. [Miniconda3](https://www.anaconda.com/docs/getting-started/miniconda/main) is a lightweight version of Conda, often used in HPC setups due to its flexibility. 

When the term "Conda" is used in this Practice, it refers to miniconda3. They have very similar functions and syntax within the command line, but it is essential to note that Miniconda3 utilizes a minimal base environment. Conda can be used to manage packages in various programming languages (R, Python, Bash, etc.) and is a robust tool. Conda files are stored in the home directory, under the folder `~/miniconda3/`. 

**Note:** `~` denotes your home directory. In our case, this is `$HOME/`. The `pwd` command run in your home directory '~' will output `$HOME/` but this is a [symbolic link](https://www.redhat.com/en/blog/linking-linux-explained). Check links with `ls -l`. 

**Note**: You are also able to change where conda stores your environments by modifying the `.condarc` file within your home directory (`~/`). If you have a researcher account with ITRSS, it is recommended to store your conda enviornments (the `/miniconda3` folder) within your `data` folder, as it has 500 GB of storage, rather than within the 50 GB within your home folder. 

### Mamba
[Mamba](https://mamba.readthedocs.io/en/latest/) is a reimplementation of Conda that is much faster. It is still a package manager and uses the same command line syntax, supports the same environment, and package management features. Mamba is written in C++ instead of Python and is therefore much faster at resolving dependencies for packages. 

Conda and Mamba can be used interchangeably, meaning any command that can be run with `conda` can be run using `mamba` instead. For example: `conda env list` would become `mamba env list`. To load the mamba module within a terminal in Jupyter Notebook or on the Hellbender desktop, simply type `module load mamba`.

Mamba environments and packages are also stored under the `~/miniconda3/ directory`, just like with Conda. This means existing environments created with Conda can be managed with Mamba without any reconfiguration.

**<u>CRUCIAL NOTE:</u> IN THIS WEEK'S MODULE, WE WILL ONLY BE USING THE `mamba` MODULE AS OF 10/14/25. THERE HAVE BEEN ISSUES WITH THE `miniconda3` MODULE ON HELLBENDER, AND ITRSS IS AWARE OF THEM. AS ITRSS WORKS TO CORRECT THESE ISSUES, `mamba` IS A DROP-IN REPLACEMENT AND ANY COMMAND THAT USES the `conda` SYNTAX FROM THE `miniconda3` MODULE HAS BEEN REPLACED.**


### Packages and Environments within Conda (miniconda3)
Conda stores packages in the folder `~/miniconda3/pkgs` and enviornments in `~/miniconda3/envs/{$EnvName}`. When a conda environment is activated, the system will run commands within that environment, ensuring that packages and dependencies from that environment are used. This means that when you're in a conda environment, the system updates the `$PATH` variable so that executables(programs) from that environment are found and used before any others. This allows for a clean workspace without package conflicts, using the programs that are installed within that conda environment. This allows users to switch between projects without fear of dependency conflicts safely. For example, say we are running two different pipelines for RNA-Seq. One requires that we use `STAR` with `R version 4.2` and `DESeq 1.38`. The other requires `R version 4.4` for use of `HISTAT2 version 2.2.1`. Conda will allow us to create two installations of `R` on the same machine, or two environments, so that we can switch between versions of R without conflict. 

### Conda Channels
Packages are distributed by conda channels, which are repositories for hosting collections of packages. The default channel, `default`, is maintained by conda. The `conda-forge` channel is community-driven and contains packages produced by many developers. The `bioconda` channel contains many bioinformatics tools. We can instruct Conda to use a specific channel, allowing us to retrieve the latest version of a package from our desired source. 

![screen](../resources/images/conda-env2-GeoHackWeek.png)

Image Credit: GeoHackWeek. For an in-depth tutorial of conda, visit [GeoHackWeek](https://geohackweek.github.io/datasharing/01-conda-tutorial/)

### Basic ~~Conda~~ Mamba Commands
**To use ~~`conda(miniconda3)`~~ `mamba` module on the HPC cluster, you must first run** `module load mamba` **in the terminal. After this, it is important to note that due to the setup of the HPC cluster, the commands** `mamba activate {$MyEnv}` **and** `mamba deactivate` **will be replaced by deprecated commands like** `source activate {$MyEnv}` **and** `source deactivate`.

**<u>We are using `mamba` in place of `conda` until the `miniconda3` module is repaired by ITRSS. In the table below, anytime `conda` is used, replace it with `mamba`. To load the conda module, use `module load miniconda3` and replace `mamba` in your command syntax with `conda`.</u>**

| Command                            | Explanation                                  | Description                                   |
|----------------------------------|----------------------------------------------|-----------------------------------------------|
| `conda create -n env_name python=3.11` | `-n`: name of the environment               | Create a new environment named `env_name` with Python 3.11     |
| `source activate env_name`         | —                                            | Activate the environment named `env_name`      |
| `source deactivate`                | —                                            | Deactivate the current active environment      |
| `conda list`                     | —                                            | List all installed packages in the active environment |
| `conda install package_name`     | —                                            | Install a package into the active environment  |
| `conda update package_name`      | —                                            | Update a package in the active environment     |
| `conda env export > env.yml`     | —                                            | Export current environment details to a YAML file      |
| `conda env create -f env.yml`    | `-f`: specify environment file (YAML)        | Create an environment from the `env.yml` file         |
| `conda remove -n env_name --all` | `-n`: environment name; `--all`: remove all packages | Remove the entire environment named `env_name`                    |
| `conda install -c channel package_name` | `-c`: specify a channel                       | Install a package from a specific channel (e.g., `conda-forge`) |
| `conda info --envs`              | —                                            | List all Conda environments                     |


<a id='yml'></a>
# Creation of Module_1 environment from Module_1.yml file
A `.yml` file is the format Conda uses to share a list of packages and channels, allowing environments to be reproduced across machines. 

For use in the rest of Module 1, we will create a conda environment with the following packages:  `jupyter`, `r-base`, `r-tidyverse`, `r-irkernel`, `r-pheatmap`, `bioconductor-deseq2`, and `blast`, so that you do not have to install them one by one. Here is a video on how to install the environment via the command line from the Module_1.yml file. Alternatively, you can run the following three commands in the code cells below, but it is unlikely to complete due to the memory constraints on your node. 

**Note: `!` can be used within a code cell to run single-line bash commands, such as `echo`, which return a value. In the event you are using `cd` or a command in which the bash shell needs to access across code cells, use `%` for single lines or `%%bash` at the top of the code cell to execute the cell as a bash script.**

#M1P1V4
<iframe width="560" height="315" src="https://www.youtube.com/embed/dwfPbXfLwsY?si=RKyqJPx9Ea90CpTj" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

In [ ]:
%%bash
#NOTE: This cell is expected to take 5-15 minutes to complete. It is also possible that it will fail due to insufficient memory.

##Load miniconda3
module load miniconda3
module load mamba

##Create environment called Module_1 from file Module_1.yml
mamba env create -f $REPO_DIR/environment/module_1.yml

##Activate environment called Module_1
source activate Module_1

##In R, run the installpec command from the package IRkernel so that the kernel will be added to your Jupyter notebooks 
##(May require restart of jupyter session through on demand)
R -e "IRkernel::installspec(name = 'Module_1', displayname = 'R (Module_1)')"

exit

To activate your environment, you will use the `source activate {$MyEnv}` command. This will update your terminal to say `({$MyEnv})[PawPrint@node# dir]` rather than just `[PawPrint@node# dir]`. This is how you know a conda environment is active. Once the `Module_1` environment is active, we will add it to the kernelspec list within Jupyter, allowing you to access the packages contained within the `Module_1' environment from within notebooks. 

<a id='make'></a>
### Creating an enviornments to fit your needs
If you would like to create an environment within conda utilizing your own choice of packages, follow this [tutorial from the Conda Documentation](https://docs.conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html).

### Global Environments
Later in the course, starting in Module 2, we will utilize global environments that were previously created/tested and are accessible within the course directory. These pre-made environments will be used later on in the course to address dependency issues and allow for the simple use of R, Python, and Bash programs that require more complex dependencies. It is essential that you, as a student, understand how environments work and how to design them, which is why we practice creating them and linking their kernel to Jupyter Notebook here. 